In [1]:
%cd practicas/

/workspace/practicas


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder
         .appName("pr503")
         .master("spark://spark-master:7077")
         .getOrCreate()
         )
 
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 08:40:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.types import StructType, StructField, BooleanType, IntegerType, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as f
from pyspark.sql import Window

schema_world = StructType([
    StructField("row_id", IntegerType(), True),
    StructField("click_datetime", TimestampType(), True),
    StructField("time_to_next_click", DoubleType(), True),
    StructField("movie_title", StringType(), True),
    StructField("movie_genres", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("title_id", StringType(), True),
    StructField("user_id", StringType(), True),
    ])

df = (spark.read
             .format("csv")
             .schema(schema_world)
             .option("header", "True")
             .load("./data/vodclickstream_uk_movies_03.csv"))
df.show(5)

+------+-------------------+------------------+--------------------+--------------------+------------+----------+----------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|release_date|  title_id|   user_id|
+------+-------------------+------------------+--------------------+--------------------+------------+----------+----------+
| 58773|2017-01-01 01:15:09|               0.0|Angus, Thongs and...|Comedy, Drama, Ro...|  2008-07-25|26bd5987e8|1dea19f6fe|
| 58774|2017-01-01 13:56:02|               0.0|The Curse of Slee...|Fantasy, Horror, ...|  2016-06-02|f26ed2675e|544dcbc510|
| 58775|2017-01-01 15:17:47|           10530.0|   London Has Fallen|    Action, Thriller|  2016-03-04|f77e500e7a|7cbcc791bf|
| 58776|2017-01-01 16:04:13|              49.0|            Vendetta|       Action, Drama|  2015-06-12|c74aec7673|ebf43c36b6|
| 58777|2017-01-01 19:16:37|               0.0|The SpongeBob Squ...|Animation, Action...|  2004-11-19|a80d6fc2aa|a57c992287|


# 1. Auditoría de telemetría Web (validación de datos)

In [4]:
ventana = ( Window.partitionBy("user_id").orderBy("click_datetime") )

df_resultado = df.select("user_id", "click_datetime").withColumn(
    "click_anterior", f.lead("click_datetime", -1).over(ventana)
).withColumn("calculated_time_to_next", (f.col("click_anterior").cast("long") - f.col("click_datetime").cast("long"))*(-1))

df_resultado.show(5)

+----------+-------------------+-------------------+-----------------------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|
+----------+-------------------+-------------------+-----------------------+
|0006ea6b5c|2017-05-19 20:21:43|               NULL|                   NULL|
|0006ea6b5c|2017-05-20 21:54:34|2017-05-19 20:21:43|                  91971|
|0006ea6b5c|2017-05-26 18:38:01|2017-05-20 21:54:34|                 506607|
|0006ea6b5c|2017-05-26 23:31:46|2017-05-26 18:38:01|                  17625|
|0006ea6b5c|2017-05-27 22:45:41|2017-05-26 23:31:46|                  83635|
+----------+-------------------+-------------------+-----------------------+
only showing top 5 rows



# 2. Deteccion de "zapping"

In [5]:
df_resultado = df_resultado.withColumn("es_zaping", f.when((f.col("click_anterior").cast("long") - f.col("click_datetime").cast("long"))*(-1) > 300 , "NO").otherwise("SI"))
df_resultado.show(5)

+----------+-------------------+-------------------+-----------------------+---------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|es_zaping|
+----------+-------------------+-------------------+-----------------------+---------+
|0006ea6b5c|2017-05-19 20:21:43|               NULL|                   NULL|       SI|
|0006ea6b5c|2017-05-20 21:54:34|2017-05-19 20:21:43|                  91971|       NO|
|0006ea6b5c|2017-05-26 18:38:01|2017-05-20 21:54:34|                 506607|       NO|
|0006ea6b5c|2017-05-26 23:31:46|2017-05-26 18:38:01|                  17625|       NO|
|0006ea6b5c|2017-05-27 22:45:41|2017-05-26 23:31:46|                  83635|       NO|
+----------+-------------------+-------------------+-----------------------+---------+
only showing top 5 rows



# 3. El rankinf de "maratones"

In [7]:
df_resultado = df_resultado.withColumn("click_date", f.split(f.col("click_datetime"), " ")[0])



ventana = (Window.partitionBy("user_Id", "click_date").orderBy("click_date"))

df_resultado = df_resultado.withColumn("row_number", f.row_number().over(ventana))

df_resultado.filter(f.col("row_number") >= 5).show(5)

+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
|   user_id|     click_datetime|     click_anterior|calculated_time_to_next|es_zaping|click_date|row_number|
+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
|0040a74430|2017-12-14 11:14:45|2017-12-14 11:06:48|                    477|       NO|2017-12-14|         5|
|004e33f215|2019-03-28 17:28:08|2019-03-28 16:36:11|                   3117|       NO|2019-03-28|         5|
|004e33f215|2019-03-28 23:44:13|2019-03-28 17:28:08|                  22565|       NO|2019-03-28|         6|
|004e33f215|2019-03-29 13:53:23|2019-03-29 13:51:43|                    100|       SI|2019-03-29|         5|
|004e33f215|2019-03-29 14:25:56|2019-03-29 13:53:23|                   1953|       NO|2019-03-29|         6|
+----------+-------------------+-------------------+-----------------------+---------+----------+----------+
only showing top 5 

# 4. Análisis de re-visualización

In [11]:
ventana = Window.partitionBy("user_id", "title_id")

df = df.withColumn("veces_vista_por_usuario", f.count("user_id").over(ventana))

df.filter(f.col("veces_vista_por_usuario") >= 3).show(10
)

+------+-------------------+------------------+--------------------+--------------------+-------------+----------+----------+-----------------------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres| release_date|  title_id|   user_id|veces_vista_por_usuario|
+------+-------------------+------------------+--------------------+--------------------+-------------+----------+----------+-----------------------+
|438764|2018-06-15 02:51:15|               0.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|438844|2018-06-15 03:01:15|               0.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|438863|2018-06-15 03:01:15|              -1.0|From Dusk till Da...|       NOT AVAILABLE|NOT AVAILABLE|4c3d7b724e|000118a755|                      3|
|578906|2018-12-30 22:05:13|               0.0|Black Mirror: Ban...|Drama, Mystery, S...|   2018-12-